# Audit: 16-bit TIFF precision (PIL vs imageio)

**目的**: 验证 `data/paired_folder.py` 用 `PIL.convert("RGB")` 读 PPR10K source TIFF 是不是把 16-bit 精度量化丢了.

**前提**: ppr10k_download.ipynb Cell 2 已经跑过, `/content/ppr10k_raw/source/` 有解压好的 .tif.

**怎么读结果**:
- 如果 PIL 每通道 ≤ 256 unique values, 但 imageio 看到 1000+ → **确认 PIL 量化, 应该改用 imageio**
- 如果两者都 ≤ 256 → 16-bit 假说错, PPR10K source 可能本质是 8-bit, 得找别的瓶颈

In [ ]:
# === Cell 1: session start (mount + pull) ===
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/LoR-LUT
!git checkout -- notebooks/
!git pull --ff-only origin main

!pip install -q imageio tifffile

In [ ]:
# === Cell 2: 测试 ===
import os, glob
import numpy as np
from PIL import Image
import imageio.v3 as iio
import torch
import torchvision.transforms.functional as TF

candidates = sorted(glob.glob('/content/ppr10k_raw/source/0_0*.tif'))
assert candidates, '❌ /content/ppr10k_raw/source/ 没文件 — 先跑 ppr10k_download.ipynb Cell 2 解压'
SAMPLE = candidates[0]
print(f'测试文件: {SAMPLE}')
print(f'文件大小: {os.path.getsize(SAMPLE)/1024:.1f} KB\n')

# === 方式 1: 当前代码用的 PIL.convert("RGB") + ToTensor ===
img_pil = Image.open(SAMPLE).convert('RGB')
print(f'[PIL] mode={img_pil.mode}, size={img_pil.size}')
arr_pil = np.array(img_pil)
tensor_pil = TF.to_tensor(img_pil)
print(f'[PIL] arr dtype={arr_pil.dtype}, shape={arr_pil.shape}, min={arr_pil.min()}, max={arr_pil.max()}')
print(f'[PIL] tensor dtype={tensor_pil.dtype}, range=[{tensor_pil.min().item():.6f}, {tensor_pil.max().item():.6f}]')

# === 方式 2: imageio 直接读保留位深 ===
arr_iio = iio.imread(SAMPLE)
print(f'\n[imageio] arr dtype={arr_iio.dtype}, shape={arr_iio.shape}, min={arr_iio.min()}, max={arr_iio.max()}')

if arr_iio.dtype == np.uint16:
    tensor_iio = torch.from_numpy(arr_iio.astype(np.float32) / 65535.0).permute(2,0,1)
elif arr_iio.dtype == np.uint8:
    tensor_iio = torch.from_numpy(arr_iio.astype(np.float32) / 255.0).permute(2,0,1)
else:
    tensor_iio = torch.from_numpy(arr_iio.astype(np.float32)).permute(2,0,1)
print(f'[imageio] tensor dtype={tensor_iio.dtype}, range=[{tensor_iio.min().item():.6f}, {tensor_iio.max().item():.6f}]')

# === 对比 1: 每通道唯一值数量 ===
print('\n' + '='*60)
print('=== 对比 1: 每通道唯一值数 (越多 = 精度越细) ===')
print('='*60)
for ch_idx, ch_name in enumerate(['R', 'G', 'B']):
    n_pil = len(np.unique(arr_pil[:,:,ch_idx]))
    if arr_iio.ndim == 3 and arr_iio.shape[2] >= 3:
        n_iio = len(np.unique(arr_iio[:,:,ch_idx]))
    else:
        n_iio = len(np.unique(arr_iio.flatten()))
    ratio = n_iio / max(n_pil, 1)
    print(f'  {ch_name}: PIL={n_pil:>5d}  imageio={n_iio:>6d}  ratio={ratio:.1f}x')

# === 对比 2: 像素值差异 ===
print('\n=== 对比 2: PIL vs imageio 在 [0,1] 空间的差异 ===')
tensor_iio_q8 = (tensor_iio * 255).round() / 255
mae_full = (tensor_pil - tensor_iio).abs().mean().item()
mae_q8 = (tensor_pil - tensor_iio_q8).abs().mean().item()
print(f'  PIL vs imageio (无量化):    MAE = {mae_full:.6f}')
print(f'  PIL vs imageio (q8 后):     MAE = {mae_q8:.6f}')
print(f'  差值 (PIL 损失精度量):       {mae_full - mae_q8:.6f}')

# === 对比 3: 16-bit 低字节信息 ===
print('\n=== 对比 3: 16-bit 原始数据真有 >256 levels 吗 ===')
if arr_iio.dtype == np.uint16:
    low_bits = arr_iio & 0xFF
    has_low = np.any(low_bits != 0)
    n_total_unique = len(np.unique(arr_iio.flatten()))
    print(f'  低 8-bit 含信息: {"✅ YES (真 16-bit)" if has_low else "❌ NO (本质 8-bit)"}')
    print(f'  整体 unique values 数: {n_total_unique}')
    print(f'  低 8-bit 非零像素比例: {(low_bits != 0).mean()*100:.1f}%')
else:
    print(f'  imageio 读到 dtype={arr_iio.dtype}, 不是 uint16')

# === 结论 ===
print('\n' + '='*60)
print('=== 结论 ===')
print('='*60)
max_pil_unique = max(len(np.unique(arr_pil[:,:,c])) for c in range(3))
if arr_iio.ndim == 3 and arr_iio.shape[2] >= 3:
    max_iio_unique = max(len(np.unique(arr_iio[:,:,c])) for c in range(3))
else:
    max_iio_unique = len(np.unique(arr_iio.flatten()))

if arr_iio.dtype == np.uint16 and max_iio_unique > 500 and max_pil_unique <= 256:
    print('✅ 确认: PPR10K source 是真 16-bit, PIL 量化到 8-bit 损失了精度')
    print(f'   PIL 单通道最多 {max_pil_unique} unique  vs  imageio {max_iio_unique}  → ~{max_iio_unique/max_pil_unique:.0f}x 损失')
    print('   → 应该改 paired_folder.py 用 imageio, 期望恢复 1-2 dB val PSNR')
elif max_iio_unique <= 256:
    print('❓ imageio 也只读到 ≤256 unique values: PPR10K 这个 zip 里的 TIF 可能本质就是 8-bit')
    print('   那 PIL 没问题, 16-bit 假说不成立, 需要 audit 别的环节')
else:
    print(f'⚠️ 不确定: PIL={max_pil_unique}, imageio={max_iio_unique}, dtype={arr_iio.dtype}')
    print('   把上面所有输出贴给 Claude 让他判断')